# `libfec_parser` quickstart

`libfec_parser` is the Python binding for [libfec](https://github.com/asg017/libfec)'s Rust `.fec` parser. This notebook walks through both APIs it ships with:

1. **`libfec_parser.parser`** (also exported at the top level as `open`/`read`) — the primary API: a streaming `FilingReader` or an eager `Filing`, with typed values (`float`, `datetime.date`) and full pandas interop. Start here.
2. **`libfec_parser.fecfile`** — a drop-in for the [`fecfile`](https://pypi.org/project/fecfile/) package (0.9.1). Rows come back as dicts keyed by column name, with the same typed values (`float`, tz-aware `datetime`) real `fecfile` produces.

The package isn't on PyPI yet, so build it from source first. From `crates/fec-py/`, `make notebook` builds the wheel and opens this notebook with everything installed. See the [README](../README.md) for other options.

## Get a filing

Every electronic filing is a public `.fec` file at `https://docquery.fec.gov/dcdev/posted/<FILING_ID>.fec`. We'll use [FEC-1721696](https://docquery.fec.gov/cgi-bin/forms/C00016683/1721696/), Pfizer Inc. PAC's monthly report for July 2023: about 260 KB with ~1,400 itemized rows.

In [1]:
import urllib.request
from pathlib import Path

FILING_ID = 1721696
candidates = [Path(f"../tests/fixtures/{FILING_ID}.fec"), Path(f"{FILING_ID}.fec")]
path = next((p for p in candidates if p.exists()), candidates[-1])
if not path.exists():
    urllib.request.urlretrieve(f"https://docquery.fec.gov/dcdev/posted/{FILING_ID}.fec", path)

print(f"{path}: {path.stat().st_size:,} bytes")

../tests/fixtures/1721696.fec: 263,105 bytes


## The native API

`libfec_parser.open()` returns a `FilingReader`: the `HDR` record and cover page are parsed eagerly, so `header`/`cover`/`cover_row` are available right away, and itemization rows are pulled lazily as you iterate — the right choice for a large filing, or when `rows(*prefixes)` lets you skip most of it. `read()` (or `Filing(source)`, the same thing) parses everything up front into a `Filing` whose `rows` is a plain `list`, which is what you want to build a `pandas.DataFrame`.

Use it as a context manager, and filter to the rows you want with `rows(*prefixes)` (matched case-insensitively, before a `Row` is even built):

In [2]:
import itertools
import libfec_parser

with libfec_parser.open(str(path)) as filing:
    print(filing.header)
    print(filing.id, "|", filing.fec_version)
    print()

    for row in itertools.islice(filing.rows("SA"), 5):
        print(row["contributor_last_name"], row["contribution_amount"], row["contribution_date"])

    cover_row = filing.cover_row

Header(fec_version='8.4', software_name='FECFile', software_version='8.4')
1721696 | 8.4

Aaronson 104.17 2023-07-14
Aaronson 104.17 2023-07-31
Aarts 20.84 2023-07-14
Aarts 20.84 2023-07-31
Adams 20.0 2023-07-14


A `Row` is a mapping from column name to a *typed* value — `contribution_amount` above is a `float`, `contribution_date` a `datetime.date`, not strings to parse yourself. `dict(row)` gives every column at once (only the populated ones shown here):

In [3]:
{k: v for k, v in dict(row).items() if v != ""}

{'form_type': 'SA11AI',
 'filer_committee_id_number': 'C00016683',
 'transaction_id': '2023071716378-3174',
 'entity_type': 'IND',
 'contributor_last_name': 'Adams',
 'contributor_first_name': 'Dorinda',
 'contributor_middle_name': 'C',
 'contributor_street_1': '66 Hudson Blvd East',
 'contributor_city': 'New York',
 'contributor_state': 'NY',
 'contributor_zip_code': '10001',
 'contribution_date': datetime.date(2023, 7, 14),
 'contribution_amount': 20.0,
 'contribution_aggregate': 280.0,
 'contributor_employer': 'Pfizer, Inc.',
 'contributor_occupation': 'National Pharmacy Business Manager'}

`cover_row` is the full cover line as a `Row` too — every column the form defines, typed the same way. This is where the cover-page totals live (`Cover.fields()` only keeps the six normalized attributes: form type, filer, coverage dates):

In [4]:
for key in [
    "form_type",
    "filer_committee_id_number",
    "committee_name",
    "report_code",
    "coverage_from_date",
    "coverage_through_date",
    "col_a_cash_on_hand_beginning_period",
    "col_a_total_receipts",
    "col_a_total_disbursements",
    "col_a_cash_on_hand_close_of_period",
]:
    print(f"{key:40} {cover_row[key]!r}")

form_type                                'F3XN'
filer_committee_id_number                'C00016683'
committee_name                           'PFIZER INC. PAC'
report_code                              'M8'
coverage_from_date                       datetime.date(2023, 7, 1)
coverage_through_date                    datetime.date(2023, 7, 31)
col_a_cash_on_hand_beginning_period      394272.48
col_a_total_receipts                     83741.93
col_a_total_disbursements                57650.0
col_a_cash_on_hand_close_of_period       420364.41


### Into pandas

`Row` registers as a `collections.abc.Mapping`, so `read(path).rows` — a plain list of them — is exactly what `pd.DataFrame` wants. Because the values are already typed, there's no `astype(float)` or `pd.to_datetime(..., format=...)` step to write: amounts are `float64` and dates are `datetime.date` objects from the moment the `DataFrame` exists.

In [5]:
import pandas as pd
from libfec_parser import read

filing = read(str(path))
df = pd.DataFrame(filing.rows)

print(df["contribution_amount"].dtype)
print(type(df["contribution_date"].dropna().iloc[0]))

receipts = df[df["form_type"].str.startswith("SA")].dropna(axis=1, how="all")
receipts[
    [
        "contributor_last_name",
        "contributor_first_name",
        "contributor_state",
        "contributor_occupation",
        "contribution_date",
        "contribution_amount",
    ]
].head()

float64
<class 'datetime.date'>


,contributor_last_name,contributor_first_name,contributor_state,contributor_occupation,contribution_date,contribution_amount
0,Aaronson,Eric,NY,"SVP, Chief Counsel IP & IPE",2023-07-14,104.17
1,Aaronson,Eric,NY,"SVP, Chief Counsel IP & IPE",2023-07-31,104.17
2,Aarts,Johanna,NY,"VP MTL, Rheumatology and New I&I Area",2023-07-14,20.84
3,Aarts,Johanna,NY,"VP MTL, Rheumatology and New I&I Area",2023-07-31,20.84
4,Adams,Dorinda,NY,National Pharmacy Business Manager,2023-07-14,20.00


(Dates land as `object`-dtype `datetime.date`, not `datetime64` — pandas doesn't auto-convert one to the other. If you need `datetime64` semantics, such as `.dt` accessors or resampling, convert explicitly with `pd.to_datetime(df["contribution_date"])`; this notebook doesn't need to.)

This PAC is funded by payroll deductions, so most people appear more than once in a month. Group by contributor to see who gave the most — the same analysis as before, now over typed columns:

In [6]:
total = receipts["contribution_amount"].sum()
print(f"{len(receipts):,} itemized receipts totaling ${total:,.2f}")
print(f"reported on the cover page:      ${filing.cover_row['col_a_individuals_itemized']:,.2f}")

(
    receipts.groupby(["contributor_last_name", "contributor_first_name", "contributor_occupation"])["contribution_amount"]
    .agg(["count", "sum"])
    .sort_values("sum", ascending=False)
    .head(10)
)

1,354 itemized receipts totaling $57,161.47
reported on the cover page:      $57,161.47


,,,count,sum
contributor_last_name,contributor_first_name,contributor_occupation,,
Susman,Sally,"Chief Corporate Affairs Officer, Execu",2,416.68
Dolsten,G.,"Chief Scientific Officer & President,",2,416.68
McDermott,Michael,"Chief Global Supply Officer, Executive",2,416.66
Power,Elizabeth,"Senior Director, Groton Site Affairs",2,416.66
Bishop-Murphy,Melissa,"Senior Director, State Government Rela",2,416.66
Mueller,Emily,Senior Director and Head of Congressio,2,416.66
Bourla,Albert,Chairman & CEO,2,416.66
Krebs,Matthew,"Senior Manager, Alliance Development",2,416.66
Johnson,Rady,"Chief Compliance,Quality & Risk Office",2,416.66


## The `fecfile` API

`libfec_parser.fecfile` is a **drop-in replacement for [`fecfile`](https://pypi.org/project/fecfile/) 0.9.1** — same keys, same order, same values and types, verified against the real package by a differential test on every fixture and a 408,162-item filing. `from_file()` parses a filing into a plain dict with four keys — `header`, `filing` (the cover page), `itemizations` (rows grouped by schedule: `"Schedule A"`, `"Schedule B"`, …), and `text`.

In [7]:
from libfec_parser import fecfile

parsed = fecfile.from_file(str(path))
parsed.keys()

dict_keys(['itemizations', 'text', 'header', 'filing'])

Values are typed the same way real `fecfile` types them: amounts parse to `float`, dates to a tz-aware `datetime` in US/Eastern, and an empty amount/date column is `None`. Pass `options={"as_strings": True}` to get the raw strings instead.

In [8]:
cover = parsed["filing"]
type(cover["col_a_total_receipts"]), cover["col_a_total_receipts"]

(float, 83741.93)

Itemizations are grouped by schedule, and each row is a dict keyed by column name:

In [9]:
{schedule: len(rows) for schedule, rows in parsed["itemizations"].items()}

{'Schedule A': 1354, 'Schedule B': 33}

In [10]:
first = parsed["itemizations"]["Schedule A"][0]

# Only show the populated columns
{k: v for k, v in first.items() if v}

{'form_type': 'SA11AI',
 'filer_committee_id_number': 'C00016683',
 'transaction_id': '2023071716378-1066',
 'entity_type': 'IND',
 'contributor_last_name': 'Aaronson',
 'contributor_first_name': 'Eric',
 'contributor_street_1': '66 Hudson Blvd East',
 'contributor_city': 'New York',
 'contributor_state': 'NY',
 'contributor_zip_code': '10001',
 'contribution_date': datetime.datetime(2023, 7, 14, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')),
 'contribution_amount': 104.17,
 'contribution_aggregate': 1458.38,
 'contributor_employer': 'Pfizer Inc',
 'contributor_occupation': 'SVP, Chief Counsel IP & IPE'}

### Only parse what you need

`filter_itemizations` takes a list of row-type prefixes and drops everything else. On a large filing (ActBlue's reports run to several gigabytes) this saves most of the memory. An empty list skips itemizations entirely, leaving just the header and cover page.

In [11]:
only_sb = fecfile.from_file(str(path), options={"filter_itemizations": ["SB"]})
{schedule: len(rows) for schedule, rows in only_sb["itemizations"].items()}

{'Schedule B': 33}

### Other entry points

- `loads(content)` parses `bytes`, a `str`, or any iterable of lines you already have in memory.
- `from_http(filing_id)` downloads from the FEC and parses in one step. It returns `None` if the filing doesn't exist (like real `fecfile`). `iter_http(filing_id)` streams the same download instead of buffering it.
- `iter_file(path)` streams a filing as `FecItem`s (`.data_type`, `.data`) — the header, then the summary, then one item per row — rather than building the whole dict, for filings too large to hold in memory at once. See below.
- `parse_header(line)` and `parse_line(line, version)` parse a single record, if you're streaming a file yourself.

One gotcha when working with lines: `.fec` fields are separated by the ASCII 28 "file separator" character, which Python's `str.splitlines()` treats as a line break. Split on `"\n"` instead.

In [12]:
from itertools import islice

for item in islice(fecfile.iter_file(str(path)), 5):
    label = item.data.get("form_type") if isinstance(item.data, dict) else item.data
    print(item.data_type, "-", label)

header - None
summary - F3XN
itemization - SA11AI
itemization - SA11AI
itemization - SA11AI


In [13]:
lines = path.read_text(encoding="latin-1").split("\n")

header, version, _ = fecfile.parse_header(lines[0])
print(header)

row = fecfile.parse_line(lines[2], version)
{k: v for k, v in row.items() if v}

{'record_type': 'HDR', 'ef_type': 'FEC', 'fec_version': '8.4', 'soft_name': 'FECFile', 'soft_ver': '8.4', 'report_id': '', 'report_number': '0', 'comment': ''}


{'form_type': 'SA11AI',
 'filer_committee_id_number': 'C00016683',
 'transaction_id': '2023071716378-1066',
 'entity_type': 'IND',
 'contributor_last_name': 'Aaronson',
 'contributor_first_name': 'Eric',
 'contributor_street_1': '66 Hudson Blvd East',
 'contributor_city': 'New York',
 'contributor_state': 'NY',
 'contributor_zip_code': '10001',
 'contribution_date': datetime.datetime(2023, 7, 14, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='America/New_York')),
 'contribution_amount': 104.17,
 'contribution_aggregate': 1458.38,
 'contributor_employer': 'Pfizer Inc',
 'contributor_occupation': 'SVP, Chief Counsel IP & IPE'}

## Next steps

- The [libfec CLI](https://github.com/asg017/libfec) can export whole election cycles of filings to SQLite or CSV, which is usually the better tool when you need more than a handful of filings.
- Column names for every form and schedule come from libfec's mappings, across FEC format versions.